# 선택 피처 실험 노트북

중요도에서 반복적으로 강했던 상위 변수 20개만 남겨서 모델 성능이 얼마나 달라지는지 확인합니다.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


ROOT = Path("/home/ms990728/소정해")
DATA_PATH = ROOT / "data/raw/cell2celltrain.csv"
RESULT_PATH = ROOT / "results/baseline_results.md"
RANDOM_STATE = 42


@dataclass
class ResultRow:
    model: str
    train_seconds: float
    roc_auc: float
    accuracy: float
    precision: float
    recall: float
    f1: float


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=["NA", ""], keep_default_na=True)
    # HandsetPrice is stored as strings because of "Unknown" placeholders.
    df["HandsetPrice"] = pd.to_numeric(
        df["HandsetPrice"].replace("Unknown", pd.NA), errors="coerce"
    )
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype("Int64")
    return df


def build_feature_sets(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, list[str], list[str]]:
    feature_df = df.drop(columns=["CustomerID", "Churn"])
    target = df["Churn"].astype(int)

    numeric_cols = [
        col for col in feature_df.columns if pd.api.types.is_numeric_dtype(feature_df[col])
    ]
    categorical_cols = [col for col in feature_df.columns if col not in numeric_cols]

    return feature_df, target, numeric_cols, categorical_cols


def build_logistic_pipeline(numeric_cols: list[str], categorical_cols: list[str]) -> Pipeline:
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ]
    )

    model = LogisticRegression(
        max_iter=10000,
        solver="saga",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])


def build_tree_pipeline(
    numeric_cols: list[str],
    categorical_cols: list[str],
    estimator,
) -> Pipeline:
    numeric_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "ordinal",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            ),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ]
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])


def evaluate_model(name: str, pipeline: Pipeline, X_train, y_train, X_test, y_test) -> ResultRow:
    start = perf_counter()
    pipeline.fit(X_train, y_train)
    train_seconds = perf_counter() - start

    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    return ResultRow(
        model=name,
        train_seconds=train_seconds,
        roc_auc=roc_auc_score(y_test, y_prob),
        accuracy=accuracy_score(y_test, y_pred),
        precision=precision_score(y_test, y_pred, zero_division=0),
        recall=recall_score(y_test, y_pred, zero_division=0),
        f1=f1_score(y_test, y_pred, zero_division=0),
    )


def format_results(results: list[ResultRow]) -> str:
    header = (
        "| Model | Train sec | ROC-AUC | Accuracy | Precision | Recall | F1 |\n"
        "|---|---:|---:|---:|---:|---:|---:|\n"
    )
    rows = [
        f"| {r.model} | {r.train_seconds:.1f} | {r.roc_auc:.4f} | {r.accuracy:.4f} | "
        f"{r.precision:.4f} | {r.recall:.4f} | {r.f1:.4f} |"
        for r in results
    ]
    return header + "\n".join(rows) + "\n"


def main() -> None:
    df = load_data(DATA_PATH)
    feature_df, target, numeric_cols, categorical_cols = build_feature_sets(df)

    X_train, X_test, y_train, y_test = train_test_split(
        feature_df,
        target,
        test_size=0.2,
        stratify=target,
        random_state=RANDOM_STATE,
    )

    pos = int(y_train.sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / max(pos, 1)

    logistic = build_logistic_pipeline(numeric_cols, categorical_cols)
    rf = build_tree_pipeline(
        numeric_cols,
        categorical_cols,
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )
    xgb = build_tree_pipeline(
        numeric_cols,
        categorical_cols,
        XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )

    results = [
        evaluate_model("Logistic Regression", logistic, X_train, y_train, X_test, y_test),
        evaluate_model("Random Forest", rf, X_train, y_train, X_test, y_test),
        evaluate_model("XGBoost", xgb, X_train, y_train, X_test, y_test),
    ]

    summary = [
        "# Baseline Experiment Results",
        "",
        f"- Dataset: `{DATA_PATH.name}`",
        f"- Rows: `{len(df):,}`",
        f"- Features used: `{len(feature_df.columns)}`",
        f"- Numeric features: `{len(numeric_cols)}`",
        f"- Categorical features: `{len(categorical_cols)}`",
        f"- Train set size: `{len(X_train):,}`",
        f"- Test set size: `{len(X_test):,}`",
        f"- Positive class weight used for XGBoost: `{scale_pos_weight:.4f}`",
        "",
        format_results(results),
        "",
        "## Notes",
        "",
        "- `HandsetPrice` was converted to numeric with `Unknown` treated as missing.",
        "- Logistic Regression uses one-hot encoding for categorical variables.",
        "- Random Forest and XGBoost use ordinal encoding for categorical variables.",
        "- Metrics are computed on the held-out 20% test split.",
        "",
    ]

    text = "\n".join(summary)
    RESULT_PATH.write_text(text, encoding="utf-8")
    print(text)


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier


RESULT_PATH = ROOT / "results/selected_features_results.md"

SELECTED_FEATURES = [
    "CurrentEquipmentDays",
    "MonthsInService",
    "RetentionCalls",
    "OverageMinutes",
    "MonthlyMinutes",
    "TotalRecurringCharge",
    "PercChangeMinutes",
    "UniqueSubs",
    "Handsets",
    "AgeHH1",
    "MadeCallToRetentionTeam",
    "HandsetRefurbished",
    "RespondsToMailOffers",
    "HandsetWebCapable",
    "CreditRating",
    "HandsetPrice",
    "DroppedBlockedCalls",
    "ActiveSubs",
    "ReferralsMadeBySubscriber",
    "AdjustmentsToCreditRating",
]


def infer_columns(feature_df: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_cols = [
        col for col in feature_df.columns if pd.api.types.is_numeric_dtype(feature_df[col])
    ]
    categorical_cols = [col for col in feature_df.columns if col not in numeric_cols]
    return numeric_cols, categorical_cols


def format_table(results: list[ResultRow]) -> str:
    header = (
        "| Model | Train sec | ROC-AUC | Accuracy | Precision | Recall | F1 |\n"
        "|---|---:|---:|---:|---:|---:|---:|\n"
    )
    rows = [
        f"| {r.model} | {r.train_seconds:.1f} | {r.roc_auc:.4f} | {r.accuracy:.4f} | "
        f"{r.precision:.4f} | {r.recall:.4f} | {r.f1:.4f} |"
        for r in results
    ]
    return header + "\n".join(rows)


def main() -> None:
    df = load_data(DATA_PATH)
    feature_df = df.drop(columns=["CustomerID", "Churn"])
    target = df["Churn"].astype(int)

    feature_df = feature_df[SELECTED_FEATURES].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        feature_df,
        target,
        test_size=0.2,
        stratify=target,
        random_state=RANDOM_STATE,
    )

    numeric_cols, categorical_cols = infer_columns(feature_df)
    pos = int(y_train.sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / max(pos, 1)

    logistic = build_logistic_pipeline(numeric_cols, categorical_cols)
    rf = build_tree_pipeline(
        numeric_cols,
        categorical_cols,
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )
    xgb = build_tree_pipeline(
        numeric_cols,
        categorical_cols,
        XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )

    results = [
        evaluate_model("Selected / Logistic Regression", logistic, X_train, y_train, X_test, y_test),
        evaluate_model("Selected / Random Forest", rf, X_train, y_train, X_test, y_test),
        evaluate_model("Selected / XGBoost", xgb, X_train, y_train, X_test, y_test),
    ]

    summary = [
        "# Selected-Feature Experiment Results",
        "",
        f"- Dataset: `{DATA_PATH.name}`",
        f"- Selected features: `{len(SELECTED_FEATURES)}`",
        f"- Train set size: `{len(X_train):,}`",
        f"- Test set size: `{len(X_test):,}`",
        "",
        "## Selected Features",
        "",
        ", ".join(f"`{f}`" for f in SELECTED_FEATURES),
        "",
        "## Results",
        "",
        format_table(results),
        "",
    ]

    text = "\n".join(summary)
    RESULT_PATH.write_text(text, encoding="utf-8")
    print(text)


if __name__ == "__main__":
    main()



# Selected-Feature Experiment Results

- Dataset: `cell2celltrain.csv`
- Selected features: `20`
- Train set size: `40,837`
- Test set size: `10,210`

## Selected Features

`CurrentEquipmentDays`, `MonthsInService`, `RetentionCalls`, `OverageMinutes`, `MonthlyMinutes`, `TotalRecurringCharge`, `PercChangeMinutes`, `UniqueSubs`, `Handsets`, `AgeHH1`, `MadeCallToRetentionTeam`, `HandsetRefurbished`, `RespondsToMailOffers`, `HandsetWebCapable`, `CreditRating`, `HandsetPrice`, `DroppedBlockedCalls`, `ActiveSubs`, `ReferralsMadeBySubscriber`, `AdjustmentsToCreditRating`

## Results

| Model | Train sec | ROC-AUC | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|
| Selected / Logistic Regression | 2.9 | 0.6080 | 0.5803 | 0.3579 | 0.5751 | 0.4413 |
| Selected / Random Forest | 1.4 | 0.6605 | 0.7117 | 0.4992 | 0.1999 | 0.2854 |
| Selected / XGBoost | 0.4 | 0.6741 | 0.6248 | 0.4037 | 0.6332 | 0.4931 |
